# CloudSecurityAuditor-v1 Live Training Evidence\n
This notebook trains a lightweight policy directly against the environment loop and generates reward/loss plots for judging evidence.

In [ ]:
# Cell 1: Dependencies (T4-friendly)
# If you're running this notebook outside the repo, first:
#   !git clone <YOUR_REPO_URL> cloud_auditor && %cd cloud_auditor

%pip -q install -U "unsloth[cu121-torch]" trl transformers accelerate bitsandbytes peft openenv fastapi uvicorn matplotlib nest_asyncio

import os, sys, time, subprocess, pathlib

# Ensure we're at repo root (contains server/ and models.py)
if not pathlib.Path("server").exists():
    raise RuntimeError("Run this notebook from the repo root (folder containing 'server/' and 'models.py').")

# Editable install so imports work
%pip -q install -e .

In [ ]:
# Cell 2: Model and Tokenizer loading (Llama-3-8B-Instruct via Unsloth 4-bit with LoRA)

import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Meta-Llama-3-8B-Instruct"
MAX_SEQ_LEN = 1024

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

# LoRA adapter (minimal, T4-friendly)
base_model = FastLanguageModel.get_peft_model(
    base_model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# TRL PPO needs a value head; wrap the Unsloth model.
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer, create_reference_model

ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(base_model)
ref_model = create_reference_model(ppo_model)

# Unsloth speedups for generation
FastLanguageModel.for_inference(ppo_model.pretrained_model)

In [ ]:
# Cell 3: PPO Trainer and OpenEnv initialization tailored to Cloud Auditor

import nest_asyncio
nest_asyncio.apply()

import requests

from cloud_auditor import CloudAuditorAction, CloudAuditorEnv

BASE_URL = "http://127.0.0.1:8000"

# Start the FastAPI OpenEnv server in the background (live environment)
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait until /docs is reachable
for _ in range(60):
    try:
        r = requests.get(BASE_URL + "/docs", timeout=1)
        if r.status_code == 200:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Server did not start on port 8000")

print("Server up at", BASE_URL)

# PPO config (small batch to fit free Colab T4)
ppo_config = PPOConfig(
    learning_rate=1e-5,
    batch_size=4,
    mini_batch_size=1,
    gradient_accumulation_steps=1,
    ppo_epochs=2,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
)

# Connect to the live environment server
env = CloudAuditorEnv(base_url=BASE_URL)

# Pull supported commands once (for prompting)
reset_result = env.reset()
SUPPORTED_COMMANDS = reset_result.observation.metadata.get("supported_commands", [])
print("Task:", reset_result.observation.task_id)
print("Supported commands:", SUPPORTED_COMMANDS)

In [ ]:
# Cell 4: Active PPO training loop (State -> LLM Action -> Env Step -> Reward -> PPO Step)

import math
from collections import deque

# Training knobs (keep small for judge-run time)
EPISODES = 30
MAX_ENV_STEPS = 8
GEN_KWARGS = dict(
    max_new_tokens=32,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,
)

rewards_log = []
loss_log = []
step_log = []

# Helper: keep output to a single CLI line
def extract_command(text: str) -> str:
    line = (text or "").strip().splitlines()[0].strip()
    # Hard stop if model emits code fences, etc.
    line = line.replace("```", "").strip()
    return line

# Helper: prompt the model to output exactly one supported command
def build_prompt(task_id: str, task_desc: str, last_output: str | None) -> str:
    cmds = "\n".join([f"- {c}" for c in SUPPORTED_COMMANDS])
    context = (last_output or "")
    if len(context) > 1200:
        context = context[-1200:]

    return (
        "You are a DevSecOps engineer operating a simulated AWS-like CLI.\n"
        f"Task ID: {task_id}\n"
        f"Task: {task_desc}\n\n"
        "Supported commands (choose one):\n"
        f"{cmds}\n\n"
        "Last command output (may be empty):\n"
        f"{context}\n\n"
        "Respond with exactly ONE command line from the supported commands. No explanation."
    )

train_step = 0

for ep in range(EPISODES):
    reset = env.reset().observation
    last_output = reset.command_output

    for t in range(MAX_ENV_STEPS):
        prompt = build_prompt(reset.task_id, reset.task_description, last_output)

        query_toks = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).input_ids.to(device)

        # Generate a command (response only)
        with torch.no_grad():
            response_toks = ppo_trainer.generate(query_toks, **GEN_KWARGS)

        # Decode only the newly generated part
        gen_text = tokenizer.decode(response_toks[0][query_toks.shape[-1]:], skip_special_tokens=True)
        command = extract_command(gen_text)

        step_res = env.step(CloudAuditorAction(command=command)).observation
        reward = float(step_res.reward)

        # PPO update
        stats = ppo_trainer.step(
            queries=[query_toks[0]],
            responses=[response_toks[0][query_toks.shape[-1]:]],
            rewards=[reward],
        )

        # Pick a robust loss key (varies by TRL version)
        loss_val = None
        for k in [
            "ppo/loss/total",
            "ppo/loss/policy",
            "loss",
        ]:
            if k in stats:
                loss_val = float(stats[k])
                break
        if loss_val is None:
            # As a fallback, log KL (still shows training signal movement)
            loss_val = float(stats.get("objective/kl", math.nan))

        step_log.append(train_step)
        rewards_log.append(reward)
        loss_log.append(loss_val)

        train_step += 1
        last_output = step_res.command_output

        if (t == 0) or step_res.done:
            print(
                f"ep={ep:02d} t={t:02d} task={step_res.task_id:<22} "
                f"cmd='{command}' reward={reward:+.3f} score={step_res.task_score:.3f} status={step_res.status}"
            )

        if step_res.done:
            break

In [ ]:
# Cell 5: Matplotlib plotting for Rewards and Loss (training step vs reward/loss)

from pathlib import Path
import csv
import matplotlib.pyplot as plt

out_dir = Path("artifacts/trl_ppo_training")
out_dir.mkdir(parents=True, exist_ok=True)

metrics_path = out_dir / "training_metrics.csv"
with metrics_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["train_step", "reward", "loss"])
    for s, r, l in zip(step_log, rewards_log, loss_log):
        w.writerow([s, r, l])

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(step_log, rewards_log, label="step reward", alpha=0.7)
plt.xlabel("training step (PPO updates)")
plt.ylabel("reward")
plt.title("Cloud Auditor: Reward vs Training Step")
plt.grid(alpha=0.25)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(step_log, loss_log, color="tab:red", label="PPO loss (or KL fallback)", alpha=0.7)
plt.xlabel("training step (PPO updates)")
plt.ylabel("loss")
plt.title("Cloud Auditor: Loss vs Training Step")
plt.grid(alpha=0.25)
plt.legend()

plt.tight_layout()
plot_path = out_dir / "training_curves.png"
plt.savefig(plot_path, dpi=180)
plt.show()

print("saved:", metrics_path)
print("saved:", plot_path)

# Clean shutdown (optional)
try:
    env.close()
except Exception:
    pass
try:
    server_proc.terminate()
except Exception:
    pass